# Huấn luyện mô hình Imitation Learning (IL) — Điều khiển bám làn + đèn tín hiệu (CARLA)

**Đồ án:** Nghiên cứu ứng dụng học tăng cường sâu (DRL) điều khiển xe bám làn mô phỏng trong môi trường CARLA.
Notebook này là bước 2/3 — warm-start policy network cho DRL (bước 3, xem `drl_training/`).

**Input (đưa vào model)**: `seg_label` (segmentation, one-hot 13 lớp) + đặc trưng số
(`speed_mps`, `yaw_rate_rps`, `previous_steer`, `previous_longitudinal`, `speed_limit_kmh`,
`traffic_light_state` one-hot 4 lớp).
**Output**: `[steer, longitudinal]` ∈ [-1, 1] (longitudinal âm = phanh, dương = ga).
**Phạm vi**: bám làn **+ xử lý đèn tín hiệu** (dừng khi đỏ) → giữ `traffic_light_state`, 4 nhãn:
`green`, `yellow`, `red`, `unknown`.

> ⚠️ **Hợp đồng quan sát (đã sửa):** `lane_offset_m`, `heading_error_rad`, `is_junction`
> **không** được đưa vào model — chỉ dùng để chẩn đoán ở mục 11 (trả về riêng dưới dạng
> `aux`). Lý do: (1) đây là các đại lượng dùng làm **reward** ở bước DRL, để lọt vào input
> sẽ khiến policy học cách "đọc" trực tiếp sai số thay vì học nhìn ảnh segmentation
> (leakage); (2) DRL observation không có các cột này — nếu IL vẫn dùng thì checkpoint
> warm-start sẽ lệch shape/ngữ nghĩa so với actor DRL. Xem
> `docs/csv_fields_by_task.md` và `docs/manual_thu_thap_du_lieu.md` mục 9.2.


## 1. Cài đặt & Import

In [ ]:
# !pip install segmentation-models-pytorch albumentations opencv-python-headless torch torchvision pandas tqdm matplotlib scikit-learn

import os, random, time
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import albumentations as A
from albumentations.pytorch import ToTensorV2
import segmentation_models_pytorch as smp
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
print("Torch:", torch.__version__, "| CUDA:", torch.cuda.is_available())


## 2. Cấu hình

**Chỉnh `CONTROL_CSV_PATH` cho khớp file log thật.**

In [ ]:
CONTROL_CSV_PATH = "./driving_log.csv"

NUM_CLASSES = 13
IMAGE_HEIGHT = 384
IMAGE_WIDTH = 480

EPISODE_COL = "session_id"
VAL_FRACTION = 0.15

# === Hop dong quan sat (PHAI khop voi DRL - xem `drl_training/policy/observation.py`) ===
# Day la buoc 2/3, warm-start cho DRL (buoc 3). KHONG dua lane_offset_m / heading_error_rad /
# is_junction vao input model: day la cac dai luong dung de tinh REWARD o buoc DRL, khong
# phai observation (xem docs/csv_fields_by_task.md va
# docs/manual_thu_thap_du_lieu.md muc 9.2 "Khong dua lane_offset_m va heading_error_rad
# vao policy observation"). Neu de lot vao observation: (1) model hoc cach "doc" truc tiep
# sai so thay vi hoc nhin anh segmentation - ro ri nhan (leakage); (2) actor DRL warm-start
# tu checkpoint nay se khong tuong thich vi DRL khong co cac cot nay trong input.
CONTINUOUS_COLS = ["speed_mps", "yaw_rate_rps", "speed_limit_kmh"]
RAW_ACTION_COLS = ["previous_steer", "previous_longitudinal"]   # da trong [-1,1], giu nguyen
FLIP_SIGN_COLS = ["yaw_rate_rps"]  # dao dau khi lat anh (cac cot con lai khong phu thuoc trai/phai)

# Cac cot CHI dung de danh gia/chan doan (khong dua vao model) - Dataset tra ve rieng duoi
# dang "aux" cho muc 11 (danh gia chi tiet), khong lan vao scalar input cua model.
AUX_COLS = ["lane_offset_m", "heading_error_rad", "is_junction"]

# Vocab CO DINH - khong suy ra tu data de tranh lech giua cac lan chay / giua train-val
TRAFFIC_LIGHT_VOCAB = ["green", "yellow", "red", "unknown"]

def normalize_traffic_light(value) -> str:
    v = str(value).strip().lower()
    return v if v in TRAFFIC_LIGHT_VOCAB else "unknown"

SCALAR_FEATURE_DIM = len(CONTINUOUS_COLS) + len(RAW_ACTION_COLS) + len(TRAFFIC_LIGHT_VOCAB)
# Thu tu ghep vector -> phia DRL (`policy/observation.py`) phai build DUNG thu tu nay.
SCALAR_FEATURE_ORDER = list(CONTINUOUS_COLS) + list(RAW_ACTION_COLS) + \
    ["traffic_light_%s" % v for v in TRAFFIC_LIGHT_VOCAB]

BATCH_SIZE = 32
EPOCHS = 30
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
GRAD_CLIP_NORM = 1.0
USE_AMP = torch.cuda.is_available()
NUM_WORKERS = 4
PERSISTENT_WORKERS = NUM_WORKERS > 0
EARLY_STOP_PATIENCE = 8

STEER_LOSS_WEIGHT = 2.0
LONGITUDINAL_LOSS_WEIGHT = 1.0
HUBER_BETA = 0.15  # SmoothL1: MSE khi |err|<beta, MAE ngoai do -> ben hon voi cac khung lai
                    # gap/hoi phuc lan (kieu DAgger) hiem gap nhung sai so lon.

# Oversample cac session/frame co den vang/do (hiem hon nhieu so voi xanh/unknown)
USE_TRAFFIC_LIGHT_OVERSAMPLING = True

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
STEER_CHECKPOINT_PATH = "best_il_model.pth"

# Nang cao: True neu muon train tren segmentation DU DOAN (tu model notebook 1) thay vi
# ground-truth - khop dung phan phoi luc inference that. Can cot "rgb_path" trong CSV.
USE_PREDICTED_SEGMENTATION = False
SEG_CHECKPOINT_PATH = "best_carla_segmentation.pth"
PREDICTED_MASK_DIR = "./predicted_seg_masks"

PALETTE = np.array([
    [0,0,0],[70,70,70],[190,153,153],[250,170,160],[220,20,60],[153,153,153],
    [157,234,50],[128,64,128],[244,35,232],[107,142,35],[0,0,142],[102,102,156],[220,220,0],
], dtype=np.uint8)


def seed_worker(worker_id):
    """Reseed `random`/numpy rieng cho tung DataLoader worker.

    Khong co ham nay thi torch chi tu seed lai RNG cua chinh no cho moi worker - RNG cua
    `random`/`numpy` bi nhan ban GIONG HET nhau giua cac worker. Nghiem trong nhat tren
    Windows/macOS (multiprocessing start method = spawn): moi worker import lai toan bo
    module va chay lai `random.seed(SEED)` o dau file => TAT CA worker sinh cung mot chuoi
    quyet dinh flip/augment. Ket qua la augmentation kem da dang hon nhieu so voi tuong.
    """
    worker_seed = torch.initial_seed() % (2 ** 32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)


DATALOADER_GENERATOR = torch.Generator()
DATALOADER_GENERATOR.manual_seed(SEED)


## 3. Nạp dữ liệu

In [ ]:
df = pd.read_csv(CONTROL_CSV_PATH)
required = ["session_id","seg_label_path","speed_mps","yaw_rate_rps","previous_steer",
            "previous_longitudinal","lane_offset_m","heading_error_rad","speed_limit_kmh",
            "traffic_light_state","is_junction","steer","longitudinal"]
missing = [c for c in required if c not in df.columns]
assert not missing, f"CSV thiếu cột: {missing}. Cột hiện có: {list(df.columns)}"

df["traffic_light_state"] = df["traffic_light_state"].apply(normalize_traffic_light)

print(f"Tổng {len(df)} mẫu, {df['session_id'].nunique()} session")
print(df["traffic_light_state"].value_counts())
df.head()


### (Tuỳ chọn nâng cao) Dùng segmentation dự đoán thay ground-truth

Chỉ chạy nếu `USE_PREDICTED_SEGMENTATION = True` và CSV có cột `rgb_path`.

In [ ]:
if USE_PREDICTED_SEGMENTATION:
    assert "rgb_path" in df.columns, "Cần cột 'rgb_path' trong CSV."
    os.makedirs(PREDICTED_MASK_DIR, exist_ok=True)
    infer_tf = A.Compose([
        A.Resize(height=IMAGE_HEIGHT, width=IMAGE_WIDTH, interpolation=cv2.INTER_LINEAR),
        A.Normalize(mean=(0.485,0.456,0.406), std=(0.229,0.224,0.225)), ToTensorV2(),
    ])
    seg_model = smp.DeepLabV3Plus(encoder_name="resnet34", encoder_weights=None, in_channels=3, classes=NUM_CLASSES)
    ckpt = torch.load(SEG_CHECKPOINT_PATH, map_location=DEVICE)
    seg_model.load_state_dict(ckpt["model_state_dict"] if "model_state_dict" in ckpt else ckpt)
    seg_model.to(DEVICE).eval()
    for p in seg_model.parameters(): p.requires_grad = False

    predicted_paths = []
    with torch.no_grad():
        for rgb_path in tqdm(df["rgb_path"], desc="Sinh segmentation dự đoán"):
            out_path = os.path.join(PREDICTED_MASK_DIR, os.path.basename(rgb_path).replace(".png", "_pred.png"))
            if not os.path.exists(out_path):
                img = cv2.cvtColor(cv2.imread(rgb_path, cv2.IMREAD_COLOR), cv2.COLOR_BGR2RGB)
                t = infer_tf(image=img)["image"].unsqueeze(0).to(DEVICE)
                with torch.cuda.amp.autocast(enabled=USE_AMP):
                    out = seg_model(t)
                cv2.imwrite(out_path, torch.argmax(out, dim=1)[0].to(torch.uint8).cpu().numpy())
            predicted_paths.append(out_path)
    df["seg_label_path"] = predicted_paths
    del seg_model
    if DEVICE == "cuda": torch.cuda.empty_cache()
    print("Đã chuyển sang segmentation dự đoán.")
else:
    print("Dùng ground-truth segmentation.")


## 4. Chia train/val theo `session_id`

In [ ]:
def split_by_session(df):
    sessions = df[EPISODE_COL].unique().tolist()
    random.Random(SEED).shuffle(sessions)
    n_val = max(1, int(len(sessions) * VAL_FRACTION))
    val_s = set(sessions[:n_val])
    return (df[~df[EPISODE_COL].isin(val_s)].reset_index(drop=True),
            df[df[EPISODE_COL].isin(val_s)].reset_index(drop=True))

train_df, val_df = split_by_session(df)
print(f"Train: {len(train_df)} mẫu ({train_df[EPISODE_COL].nunique()} session) | "
      f"Val: {len(val_df)} mẫu ({val_df[EPISODE_COL].nunique()} session)")
print("\nPhân bố đèn tín hiệu (train):"); print(train_df["traffic_light_state"].value_counts(normalize=True).round(3))


## 5. Chuẩn hoá đặc trưng số

Z-score fit **chỉ trên train** để tránh rò rỉ dữ liệu.

In [ ]:
def compute_norm_stats(df, cols):
    return {c: (float(df[c].mean()), float(df[c].std()) + 1e-6) for c in cols}

norm_stats = compute_norm_stats(train_df, CONTINUOUS_COLS)
for c, (m, s) in norm_stats.items():
    print(f"{c}: mean={m:.4f} std={s:.4f}")


## 6. Dataset

One-hot segmentation; lật ngang kèm đảo dấu đồng bộ (`steer`, `previous_steer`, `yaw_rate_rps`, `lane_offset_m`, `heading_error_rad`).

In [ ]:
class SteeringDataset(Dataset):
    def __init__(self, dataframe, augment):
        self.df = dataframe.reset_index(drop=True)
        self.augment = augment

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        mask = cv2.imread(row["seg_label_path"], cv2.IMREAD_GRAYSCALE)
        if mask is None:
            raise FileNotFoundError(row["seg_label_path"])
        if mask.shape[0] != IMAGE_HEIGHT or mask.shape[1] != IMAGE_WIDTH:
            mask = cv2.resize(mask, (IMAGE_WIDTH, IMAGE_HEIGHT), interpolation=cv2.INTER_NEAREST)

        steer = float(row["steer"]); longitudinal = float(row["longitudinal"])
        previous_steer = float(row["previous_steer"])
        yaw_rate = float(row["yaw_rate_rps"])
        lane_offset = float(row["lane_offset_m"])        # chi dung cho aux (danh gia)
        heading_error = float(row["heading_error_rad"])  # chi dung cho aux (danh gia)

        if self.augment and random.random() < 0.5:
            mask = np.ascontiguousarray(np.fliplr(mask))
            steer, previous_steer = -steer, -previous_steer
            yaw_rate = -yaw_rate
            lane_offset, heading_error = -lane_offset, -heading_error

        mask_tensor = F.one_hot(torch.from_numpy(mask.astype(np.int64)), num_classes=NUM_CLASSES).permute(2, 0, 1).float()

        cont_src = {"speed_mps": float(row["speed_mps"]), "yaw_rate_rps": yaw_rate,
                    "speed_limit_kmh": float(row["speed_limit_kmh"])}
        cont_vals = [(cont_src[c] - norm_stats[c][0]) / norm_stats[c][1] for c in CONTINUOUS_COLS]
        raw_vals = [previous_steer, float(row["previous_longitudinal"])]
        tl_onehot = [1.0 if row["traffic_light_state"] == v else 0.0 for v in TRAFFIC_LIGHT_VOCAB]
        scalar = np.array(cont_vals + raw_vals + tl_onehot, dtype=np.float32)

        # aux: CHI de chan doan o muc 11, khong dua vao model (xem giai thich o cell cau hinh).
        aux = np.array([lane_offset, heading_error, float(row["is_junction"])], dtype=np.float32)

        target = torch.tensor([steer, longitudinal], dtype=torch.float32)
        return mask_tensor, torch.from_numpy(scalar), torch.from_numpy(aux), target


train_ds = SteeringDataset(train_df, augment=True)
val_ds = SteeringDataset(val_df, augment=False)

# worker_init_fn + generator: xem ham seed_worker() o cell cau hinh - tranh cac DataLoader
# worker sinh cung mot chuoi augmentation (bug pho bien, nghiem trong hon tren Windows/spawn).
_loader_kwargs = dict(
    num_workers=NUM_WORKERS, pin_memory=True, generator=DATALOADER_GENERATOR,
    worker_init_fn=seed_worker if NUM_WORKERS > 0 else None,
    persistent_workers=PERSISTENT_WORKERS,
)
if NUM_WORKERS > 0:
    _loader_kwargs["prefetch_factor"] = 4

if USE_TRAFFIC_LIGHT_OVERSAMPLING:
    freq = train_df["traffic_light_state"].value_counts(normalize=True)
    sample_weights = train_df["traffic_light_state"].map(lambda v: 1.0 / freq[v]).values
    sampler = WeightedRandomSampler(sample_weights, num_samples=len(train_df), replacement=True,
                                     generator=DATALOADER_GENERATOR)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler, drop_last=True, **_loader_kwargs)
    print("Dùng WeightedRandomSampler — oversample nhãn đèn hiếm (yellow/red).")
else:
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=True, **_loader_kwargs)

val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, **_loader_kwargs)


## 7. Trực quan hóa dữ liệu mẫu

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for i in range(3):
    m, s, aux, t = train_ds[random.randint(0, len(train_ds) - 1)]
    axes[i].imshow(PALETTE[torch.argmax(m, dim=0).numpy().clip(0, NUM_CLASSES - 1)])
    axes[i].set_title(f"steer={t[0]:.2f} long={t[1]:.2f}"); axes[i].axis("off")
plt.tight_layout(); plt.show()

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].hist(df["steer"], bins=50); axes[0].set_title("Phân bố Steer")
axes[1].hist(df["longitudinal"], bins=50); axes[1].set_title("Phân bố Longitudinal")
df["traffic_light_state"].value_counts().reindex(TRAFFIC_LIGHT_VOCAB).plot(kind="bar", ax=axes[2])
axes[2].set_title("Phân bố Traffic Light State")
plt.tight_layout(); plt.show()


## 8. Model — 2 nhánh (CNN + MLP)

In [ ]:
class SteeringNet(nn.Module):
    """CNN (segmentation one-hot) + MLP (scalar), 2 nhánh -> [steer, longitudinal].

    QUAN TRỌNG: kiến trúc và TÊN thuộc tính (`conv`, `pool`, `cnn_fc`, `scalar_mlp`, `head`)
    phải khớp với `drl_training/policy/backbone.py` + `actor_critic.py` — đó là nơi actor PPO
    nạp lại trọng số warm-start từ checkpoint này (`best_il_model.pth`). Đổi kiến trúc ở đây
    thì phải cập nhật đồng bộ bên DRL, nếu không loader sẽ báo lỗi shape mismatch.
    """
    def __init__(self, in_channels=NUM_CLASSES, num_scalar_features=SCALAR_FEATURE_DIM):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, 24, 5, 2, 2), nn.BatchNorm2d(24), nn.ELU(),
            nn.Conv2d(24, 36, 5, 2, 2), nn.BatchNorm2d(36), nn.ELU(),
            nn.Conv2d(36, 48, 5, 2, 2), nn.BatchNorm2d(48), nn.ELU(),
            nn.Conv2d(48, 64, 3, 2, 1), nn.BatchNorm2d(64), nn.ELU(),
            nn.Conv2d(64, 64, 3, 1, 1), nn.BatchNorm2d(64), nn.ELU(),
        )
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.cnn_fc = nn.Sequential(nn.Linear(64, 64), nn.ELU())
        self.scalar_mlp = nn.Sequential(nn.Linear(num_scalar_features, 32), nn.ELU(), nn.Linear(32, 32), nn.ELU())
        self.head = nn.Sequential(
            nn.Linear(64 + 32, 64), nn.ELU(), nn.Dropout(0.3),
            nn.Linear(64, 32), nn.ELU(), nn.Dropout(0.2),
            nn.Linear(32, 2),
        )

    def forward(self, seg_map, scalar_features):
        x_img = self.cnn_fc(self.pool(self.conv(seg_map)).flatten(1))
        x_sca = self.scalar_mlp(scalar_features)
        out = self.head(torch.cat([x_img, x_sca], dim=1))
        return torch.tanh(out)  # tanh theo tung phan tu -> tuong duong tach steer/long rieng


def control_loss(pred, target):
    # SmoothL1 (Huber) thay MSE thuan: ben hon voi cac khung lai gap/hoi phuc lan hiem gap
    # nhung sai so lon (kieu DAgger), tranh vai mau ngoai lai chi phoi gradient ca batch.
    steer_l = F.smooth_l1_loss(pred[:, 0], target[:, 0], beta=HUBER_BETA)
    long_l = F.smooth_l1_loss(pred[:, 1], target[:, 1], beta=HUBER_BETA)
    return STEER_LOSS_WEIGHT * steer_l + LONGITUDINAL_LOSS_WEIGHT * long_l


def build_param_groups(module, weight_decay):
    """Tach bias/BatchNorm ra khoi weight decay - thuc hanh chuan (vd. AdamW trong timm/
    torchvision reference training), tranh phat norm layer va bias (khong de overfit,
    bi regularize sai cach se lam giam hieu nang thay vi tang generalization)."""
    decay, no_decay = [], []
    for name, param in module.named_parameters():
        if not param.requires_grad:
            continue
        if param.ndim <= 1 or name.endswith(".bias"):
            no_decay.append(param)
        else:
            decay.append(param)
    return [
        {"params": decay, "weight_decay": weight_decay},
        {"params": no_decay, "weight_decay": 0.0},
    ]


model = SteeringNet().to(DEVICE)

N_GPUS = torch.cuda.device_count()
if N_GPUS > 1:
    print(f"Phát hiện {N_GPUS} GPU -> dùng nn.DataParallel")
    model = nn.DataParallel(model)

def get_state_dict(m):
    return m.module.state_dict() if hasattr(m, "module") else m.state_dict()

optimizer = torch.optim.AdamW(build_param_groups(model, WEIGHT_DECAY), lr=LEARNING_RATE)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)
print(f"Số tham số: {sum(p.numel() for p in model.parameters())/1e6:.3f}M")


## 9. Vòng lặp huấn luyện

In [ ]:
def run_epoch(loader, train):
    model.train() if train else model.eval()
    total_loss, total_mae, n = 0.0, np.zeros(2), 0
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for masks, scalars, _aux, targets in loader:
            masks = masks.to(DEVICE, non_blocking=True)
            scalars = scalars.to(DEVICE, non_blocking=True)
            targets = targets.to(DEVICE, non_blocking=True)
            if train: optimizer.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast(enabled=USE_AMP):
                preds = model(masks, scalars)
                loss = control_loss(preds, targets)
            if train:
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
                scaler.step(optimizer); scaler.update()
            total_loss += loss.item()
            total_mae += torch.abs(preds - targets).mean(dim=0).detach().cpu().numpy()
            n += 1
    return total_loss / n, total_mae / n


In [ ]:
history = {"train_loss": [], "val_loss": [], "val_steer_mae": [], "val_long_mae": [], "epoch_time": [], "lr": []}
best_val_loss = float("inf"); epochs_no_improve = 0
training_start = time.time()

pbar = tqdm(range(EPOCHS), desc="Training IL")
for epoch in pbar:
    t0 = time.time()
    train_loss, _ = run_epoch(train_loader, True)
    val_loss, val_mae = run_epoch(val_loader, False)
    scheduler.step()

    history["train_loss"].append(train_loss); history["val_loss"].append(val_loss)
    history["val_steer_mae"].append(val_mae[0]); history["val_long_mae"].append(val_mae[1])
    history["epoch_time"].append(time.time() - t0); history["lr"].append(optimizer.param_groups[0]["lr"])
    pbar.set_postfix({"train_loss": f"{train_loss:.4f}", "val_loss": f"{val_loss:.4f}", "steer_mae": f"{val_mae[0]:.4f}"})

    if val_loss < best_val_loss:
        best_val_loss = val_loss; epochs_no_improve = 0
        torch.save({
            "epoch": epoch, "model_state_dict": get_state_dict(model),
            "optimizer_state_dict": optimizer.state_dict(), "scheduler_state_dict": scheduler.state_dict(),
            "best_val_loss": best_val_loss, "num_classes": NUM_CLASSES,
            "image_height": IMAGE_HEIGHT, "image_width": IMAGE_WIDTH,
            "scalar_feature_dim": SCALAR_FEATURE_DIM, "continuous_cols": CONTINUOUS_COLS,
            "raw_action_cols": RAW_ACTION_COLS, "scalar_feature_order": SCALAR_FEATURE_ORDER,
            "norm_stats": norm_stats, "traffic_light_vocab": TRAFFIC_LIGHT_VOCAB,
        }, STEER_CHECKPOINT_PATH)
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= EARLY_STOP_PATIENCE:
            print(f"Dừng sớm ở epoch {epoch+1}"); break

print(f"Hoàn tất. Best Val Loss: {best_val_loss:.4f} | Tổng thời gian: {(time.time()-training_start)/60:.1f} phút | {STEER_CHECKPOINT_PATH}")


> Checkpoint lưu kèm `norm_stats`, `continuous_cols`, `raw_action_cols`,
> `scalar_feature_order` và `traffic_light_vocab` — bắt buộc nạp lại đúng các giá trị này lúc
> inference/demo, không tính lại từ dữ liệu mới. `drl_training/policy/observation.py` đọc
> trực tiếp các trường này từ checkpoint để dựng scalar vector đúng thứ tự khi warm-start
> actor PPO, thay vì hard-code lại — tránh hai bên (IL notebook trên Kaggle và DRL module
> chạy local) lệch nhau khi đặc trưng thay đổi sau này.


## 10. Biểu đồ huấn luyện

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].plot(history["train_loss"], label="Train"); axes[0].plot(history["val_loss"], label="Val")
axes[0].set_title("Loss theo epoch"); axes[0].legend()
axes[1].plot(history["val_steer_mae"], label="Steer MAE"); axes[1].plot(history["val_long_mae"], label="Longitudinal MAE")
axes[1].set_title("MAE theo epoch (Val)"); axes[1].legend()
axes[2].plot(history["lr"], color="orange"); axes[2].set_title("LR schedule (Cosine)")
plt.tight_layout(); plt.show()
print(f"Thời gian trung bình/epoch: {np.mean(history['epoch_time']):.1f}s")


## 11. Đánh giá chi tiết trên tập Validation

Gồm: scatter dự đoán-thực tế, phân bố sai số, và **MAE theo từng trạng thái đèn tín hiệu**
(quan trọng để kiểm tra model có phản ứng đúng lúc đèn đỏ hay không — vì đây là nhãn hiếm).

In [ ]:
ckpt = torch.load(STEER_CHECKPOINT_PATH, map_location=DEVICE)
model.load_state_dict(ckpt["model_state_dict"]); model.eval()

all_preds, all_targets, all_aux = [], [], []
with torch.no_grad():
    for masks, scalars, aux, targets in val_loader:
        preds = model(masks.to(DEVICE), scalars.to(DEVICE)).cpu().numpy()
        all_preds.append(preds); all_targets.append(targets.numpy()); all_aux.append(aux.numpy())
all_preds = np.concatenate(all_preds); all_targets = np.concatenate(all_targets)
all_aux = np.concatenate(all_aux)  # cols: lane_offset_m, heading_error_rad, is_junction
all_tl = val_df["traffic_light_state"].values[:len(all_preds)]

labels = ["Steer", "Longitudinal"]
fig, axes = plt.subplots(1, 2, figsize=(11, 5))
for i, lb in enumerate(labels):
    axes[i].scatter(all_targets[:, i], all_preds[:, i], alpha=0.3, s=8)
    lims = [min(all_targets[:,i].min(), all_preds[:,i].min()), max(all_targets[:,i].max(), all_preds[:,i].max())]
    axes[i].plot(lims, lims, "r--", lw=1); axes[i].set_xlabel("Thực tế"); axes[i].set_ylabel("Dự đoán"); axes[i].set_title(lb)
plt.tight_layout(); plt.show()

errors = all_preds - all_targets
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(errors[:, 0], bins=50); axes[0].set_title("Phân bố sai số Steer")
axes[1].hist(errors[:, 1], bins=50); axes[1].set_title("Phân bố sai số Longitudinal")
plt.tight_layout(); plt.show()

mae_by_tl = pd.DataFrame({"tl": all_tl, "steer_ae": np.abs(errors[:, 0]), "long_ae": np.abs(errors[:, 1])})
mae_by_tl = mae_by_tl.groupby("tl")[["steer_ae", "long_ae"]].mean().reindex(TRAFFIC_LIGHT_VOCAB)
mae_by_tl.plot(kind="bar", figsize=(8, 4), title="MAE theo trạng thái đèn tín hiệu")
plt.tight_layout(); plt.show()
print(mae_by_tl)

# Sai số theo mức lệch làn (aux — KHÔNG đưa vào model, chỉ dùng để chẩn đoán ở đây).
# Kỳ vọng: steer_ae tăng dần khi |lane_offset_m| lớn — đây là vùng model phải lái mạnh để
# hồi phục, khó nhất, và cũng là vùng cần nhiều dữ liệu recovery/DAgger nhất nếu MAE cao.
lane_offset_bins = pd.cut(np.abs(all_aux[:, 0]), bins=[0, 0.15, 0.4, 0.8, np.inf],
                           labels=["<0.15m", "0.15-0.4m", "0.4-0.8m", ">0.8m"])
mae_by_offset = pd.DataFrame({"bin": lane_offset_bins, "steer_ae": np.abs(errors[:, 0])})
mae_by_offset = mae_by_offset.groupby("bin", observed=False)["steer_ae"].agg(["mean", "count"])
print("\nMAE steer theo |lane_offset_m| (chỉ để chẩn đoán, KHÔNG dùng làm input model):")
print(mae_by_offset)

mae_final = np.abs(errors).mean(axis=0)
print(f"\nMAE tổng — Steer: {mae_final[0]:.4f} | Longitudinal: {mae_final[1]:.4f}")
print(f"✅ Model IL sẵn sàng tại: {STEER_CHECKPOINT_PATH}")
